# 같은 12프레임을 다시 찍으면 체간 기울기 변화가 얼마나 달라질까?

21번 실험의 선수, 이륙과 무릎 통과 프레임, 가까운 쪽과 각도 계산식을 그대로 사용합니다. 이번에 다시 정하는 것은 어깨와 골반의 위치, 그리고 각 점의 가시성입니다.

**Run All**을 눌러 시작하세요. 판독 화면에는 섞어 놓은 원본 한 장과 선수 기준의 가까운 쪽이 표시됩니다. 촬영 장소나 인물은 알아볼 수 있으므로 이전 기록·각도표를 열지 않은 상태에서 진행하세요.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from wlcv.config import ANNOTATIONS, ROOT, SPLIT, VIDEOS
from wlcv.annotation.trunk_repeat import blind_editor, pending_frames, repeat_frames
from wlcv.evaluation.trunk_repeat import (
    compare_repeats, group_comparison, plot_point_comparison, plot_repeats,
)

split = pd.read_csv(SPLIT)
validation = split.loc[split["role"] == "validation", ["video_id", "person_id", "group"]]
metadata = pd.read_csv(ROOT / "data/metadata/videos.csv")
videos = validation.merge(metadata[["video_id", "filename", "near_side"]], on="video_id",
                          validate="one_to_one").set_index("video_id")
folder = ANNOTATIONS / "trunk_inclination"
frames = repeat_frames(videos, folder)

## 1. 가까운 쪽 어깨와 골반 다시 찍기

어깨는 견봉, 골반은 고관절 소켓의 화면상 위치를 기준으로 합니다. 클릭하기 전에 **찍을 점**에서 가시성을 고르세요. 옷 때문에 위치를 짐작하면 **옷에 가려 추정**, 소켓 위치를 짐작하면 **소켓 위치 추정**입니다. 판단하기 어렵다면 **가려서 판독 불가**나 **위치 불확실**을 선택하세요.

한 프레임에서 두 관절을 기록하고 **다음 미완료**를 누릅니다. 각 클릭은 2차 전용 폴더에 저장됩니다. 중간에 닫아도 다시 실행하면 첫 미완료 프레임부터 이어집니다. 12장을 마치면 **마지막 셀만 다시 실행**하세요.

In [ ]:
display(blind_editor(frames, videos, folder, VIDEOS))

## 2. 두 판독 비교하기

두 회차의 같은 프레임 각도, 선수별 체간 변화량(무릎 통과 − 이륙), 변화 크기(절댓값)를 차례로 확인합니다. **크기 차이**가 양수이면 2차에서 변화의 크기가 더 크게 측정됐다는 뜻입니다. 같은 선수끼리 짝지어 그룹 평균을 냅니다. 추정점이 있는 비교는 참고값으로 구분하고, 각도를 계산할 수 없는 경우는 수치에서 제외합니다.

산점도는 두 회차의 변화 크기를 보여 줍니다. 확대 그림은 같은 프레임에서 두 판독의 각도가 가장 많이 달랐던 장면입니다. 십자는 직접점, 마름모는 추정점입니다. 이 재판독은 점을 다시 찍을 때 생기는 차이를 봅니다. 두 번 비슷하게 찍혀도 소켓의 실제 위치가 확인되는 것은 아닙니다.

In [ ]:
remaining = pending_frames(frames, folder)
if remaining:
    print(f"{len(frames) - len(remaining)}/{len(frames)}장 완료 · 모든 프레임을 기록하면 비교표가 열립니다.")
else:
    frame_table, trials = compare_repeats(videos, folder)
    groups = group_comparison(trials)
    for title, table in [("프레임별 각도와 가시성", frame_table),
                         ("선수별 두 시점 간 변화", trials), ("그룹별 변화 크기", groups)]:
        print(title)
        display(table.style.format(precision=2, na_rep="—"))

    pair = trials.set_index("선수").loc[["a05", "b02"]]
    if "비교 불가" not in pair["비교 구분"].values:
        print("a05 변화 크기 − b02 변화 크기 (양수이면 a05가 큼)")
        for number in (1, 2):
            size = pair[f"{number}차 크기 (°)"]
            print(f"{number}차 {size['a05'] - size['b02']:+.2f}°")

    for quality, group_rows in groups.groupby("비교 구분"):
        if len(group_rows) == 3 and (group_rows["비교인원"] == 2).all():
            print(f"그룹 비교 ({quality}) · 각 그룹 2명, 변화 크기 작은 순")
            for number in (1, 2):
                rank = group_rows.set_index("그룹")[f"{number}차 평균 크기 (°)"].rank(method="min")
                print(f"{number}차 순위", rank.sort_values().astype(int).to_dict())
        else:
            print(f"{quality} 그룹 평균의 비교 인원을 확인하세요. 전체 그룹 순위는 보류합니다.")

    if (trials["비교 구분"] != "비교 불가").any():
        plot_repeats(trials)
        plt.show()
    differences = frame_table["각도 차이 (2차−1차, °)"].abs().dropna()
    if not differences.empty:
        plot_point_comparison(frame_table.loc[differences.idxmax()], videos, folder, VIDEOS)
        plt.show()

    output = ROOT / "outputs/pose_baseline/trunk_repeat"
    output.mkdir(parents=True, exist_ok=True)
    for name, table in [("frames", frame_table), ("trials", trials), ("groups", groups)]:
        table.to_csv(output / f"{name}.csv", index=False)
    print("비교표 저장 위치:", output)